# v0.7 Local Training — Composer + AC + Decoder

Runs on M3 CPU. Loads ep-42 encoder checkpoint, re-inits composer to match HPO seed pattern,
trains composer → AC → decoder. No encoder retraining, no evals (run evals on GPU afterwards).

**Uses checkpoint-era code from `ep42_code/`** — the `biojepa_v0_7.py` and `training_v0_7.py`
that correspond to the saved checkpoint, not the modified versions in `biojepa/`.

Expected runtime on M3 Air: composer ~48 min, AC and decoder ~15-30 min each.

## Imports

In [1]:
import sys
import gc
import random
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt

# cloud_notebooks/ (notebook dir) takes precedence for biojepa_v0_7 / training_v0_7,
# biojepa/ parent supplies dataloader_v0_7 and config_v0_7.
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))

from biojepa_v0_7 import BioJepa, BioJepaConfig, ActionComposer, ActionComposerConfig
from training_v0_7 import (
    create_model, load_feature_banks, maybe_compile, reset_seed,
    run_composer_training, run_ac_training, train_linear_decoder,
)
from dataloader_v0_7 import ComposerLoader, TrainingLoader
from config_v0_7 import EncoderTrainingConfig, ComposerTrainingConfig, ACTrainingConfig, DecoderConfig, DataConfig


## Device & Paths — local CPU

In [2]:
SEED = 1337
torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = torch.device('cpu')
print(f'using {device}')

USE_AMP = False
USE_COMPILE = False
USE_FUSED = False

data_root = Path('~/data/jepa/v0_7').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results',
)


using cpu


## Configs — v9 HPO winner + plan changes

In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000, n_layer=6, heads=4, embed_dim=256,
    mlp_ratio=4.0, n_pre_layer=2, mask_ratio=0.766,
    gaussian_scale=2.38, film_linear_multiple=0.81,
    sim_coeff=50, std_coeff=25, cov_coeff=1,
    pert_latent_dim=128, pert_mode_dim=64,
)

# encoder_cfg unused in this run (ep-42 already trained); keep for DataConfig fields
encoder_cfg = EncoderTrainingConfig(
    epochs=50, lr=5e-5, batch_size=64, phase2_start_pct=0.8,
    context_coeff=2.0, context_ramp_pct=0.2, ema_final_momentum=1.0,
)

# v9 HPO winner, 10k epochs
composer_cfg = ComposerTrainingConfig(
    epochs=10000, lr=3.6e-4, batch_size=64,
    weight_decay=0.003, temperature=0.00126, chemical_fraction=0.1,
)

# predictor_lr reverted to 1e-4, composer_lr_mult=0.01
ac_cfg = ACTrainingConfig(
    epochs=20, predictor_lr=1e-4, batch_size=32,
    mask_anneal_pct=0.2, mask_anneal_floor=0.0,
    beta_nll_target=0.2, beta_nll_anneal_pct=0.4,
    composer_lr_mult=0.01,
)

decoder_cfg = DecoderConfig(epochs=20, lr=1e-3, batch_size=32)


## Build model + load feature banks

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)
seq_banks, target_bank = load_feature_banks(data_cfg, device)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters() if p.requires_grad):,}')
print(f'ACPredictor:     {sum(p.numel() for p in model.predictor.parameters() if p.requires_grad):,}')
print(f'Composer:        {sum(p.numel() for p in model.composer.parameters() if p.requires_grad):,}')


Loaded DNA embeddings: torch.Size([11643, 1536])
Loaded chemical embeddings: torch.Size([188, 1536])
Loaded target embeddings: torch.Size([9975, 320])
Student/Teacher: 7,979,522
ACPredictor:     9,987,072
Composer:        444,224


## Load ep-42 encoder checkpoint

Strips `_orig_mod.` prefix from the state dict keys — the checkpoint was saved on the cloud
with `USE_COMPILE=True`, so compiled submodules had that prefix. Locally we are not compiling.

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_encoder_final_epoch_42.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = {k.replace('_orig_mod.', ''): v for k, v in checkpoint['model'].items()}
keys = model.load_state_dict(state_dict)
print(f'missing keys: {len(keys.missing_keys)}')
print(f'unexpected keys: {len(keys.unexpected_keys)}')


missing keys: 0
unexpected keys: 0


## Re-init composer to match HPO seed pattern

Without this, composer weights are from `BioJepa.__init__`'s RNG state (after student/teacher consumed RNG),
which differs from HPO's composer init (directly after reset_seed). HPO convergence depends on matching this init.

In [6]:
reset_seed(SEED)

composer_init_cfg = ActionComposerConfig(
    latent_dim=model_cfg.pert_latent_dim,
    mode_dim=model_cfg.pert_mode_dim,
    heads=model_cfg.heads,
)
model.composer = ActionComposer(composer_init_cfg).to(device)

n = sum(p.numel() for p in model.composer.parameters())
print(f'composer re-initialized with HPO seed pattern: {n:,} params')


composer re-initialized with HPO seed pattern: 444,224 params


## Composer training (~48 min on M3)

In [7]:
comp_train_loader = ComposerLoader(
    batch_size=composer_cfg.batch_size,
    split='train', data_dir=data_cfg.data_root / 'pert_embd',
    device=device, seed=SEED, chemical_fraction=composer_cfg.chemical_fraction,
)
comp_val_loader = ComposerLoader(
    batch_size=composer_cfg.batch_size,
    split='val', data_dir=data_cfg.data_root / 'pert_embd',
    device=device,
)


found 1 shards for split train
  modality balancing: target chemical_fraction=10%
  adjusted total_samples=11829 (from 1 shards)
found 1 shards for split val


In [8]:
align_results = run_composer_training(
    model, comp_train_loader, comp_val_loader, seq_banks, target_bank,
    composer_cfg, device, data_cfg.checkpoint_dir,
    use_amp=USE_AMP, use_fused_optimizer=USE_FUSED,
)


Composer training: 11829 samples, 184 steps/epoch, 1840000 total steps
InfoNCE loss (temperature: 0.00126)
Step 0 | Loss: 44.83782 | LR: 1.44e-05
Step 2500 | Loss: 3.70491 | LR: 1.50e-05
Step 5000 | Loss: 3.90017 | LR: 1.69e-05
Step 7500 | Loss: 3.55647 | LR: 2.00e-05
Step 10000 | val loss: 4.6611
Step 10000 | Loss: 3.31848 | LR: 2.44e-05
Step 12500 | Loss: 3.68033 | LR: 2.99e-05
Step 15000 | Loss: 3.23304 | LR: 3.66e-05
Step 17500 | Loss: 3.44397 | LR: 4.44e-05
Step 20000 | val loss: 4.9301
Step 20000 | Loss: 3.63469 | LR: 5.32e-05
Step 22500 | Loss: 3.28586 | LR: 6.29e-05
Step 25000 | Loss: 3.33277 | LR: 7.36e-05
Step 27500 | Loss: 3.31436 | LR: 8.52e-05
Step 30000 | val loss: 4.2879
Step 30000 | Loss: 3.46802 | LR: 9.74e-05
Step 32500 | Loss: 3.58588 | LR: 1.10e-04
Step 35000 | Loss: 3.21245 | LR: 1.24e-04
Step 37500 | Loss: 3.28932 | LR: 1.38e-04
Step 40000 | val loss: 4.1850
Step 40000 | Loss: 2.88010 | LR: 1.52e-04
Step 42500 | Loss: 2.86825 | LR: 1.67e-04
Step 45000 | Loss: 3.17

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(align_results['loss_history'])
plt.yscale('log')
plt.title('Composer Training Loss')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.show()


In [ ]:
del comp_train_loader, comp_val_loader
gc.collect()
model.train()
